In [1]:
import pandas as pd
import xarray as xr
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import glob
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
# 1. Pad naar jouw Parquet-bestand
# BASE_DIR = "/Users/gjwdijk/kedro/wf-kedro/data/01_raw/parquet/heart_rate_sample/"
BASE_DIR= f'/Volumes/Extreme SSD/pq/parquet/heart_rate_sample/'
#profile_id=9
#output_path = f"{BASE_DIR}profile_id={profile_id}/"
output_path = f"{BASE_DIR}"

In [36]:
# 2. Lees het Parquet-bestand in met Pandas
# We gaan ervan uit dat er kolommen zijn voor de tijd en eventuele andere dimensies/variabelen
files = glob.glob(f"{BASE_DIR}/**/*.parquet", recursive=True)

dfs = []
n = 0
nt = len(files)
for f in files:
    print(f'{n}/{nt}')
    n += 1
    try:
        df_part = pq.read_table(f).to_pandas().drop(columns=['id', 'is_manual', 'min', 'max', 'value', 'duration'])
        if 'heart_rate_samples' in df_part.columns:
            # 1. Explodeer de kolom: elke sub-array (koppel) krijgt direct zijn eigen rij
            # Dit breekt de ongelijke rijen (van 15 en 60) direct supersnel af
            df_plat = df_part.explode('heart_rate_samples')
            
            # 2. Verwijder eventuele lege rijen (NaNs)
            df_plat = df_plat.dropna(subset=['heart_rate_samples'])
            
            if len(df_plat) > 0:
                # 3. Gebruik np.vstack om in 1x een snelle 2D NumPy-matrix te maken (C-snelheid)
                matrix = np.vstack(df_plat['heart_rate_samples'].values)
                
                # 4. Maak de kolommen direct aan door de matrix te snijden (slicing)
                # Dit kost nauwelijks tijd omdat er geen nieuwe objecten worden gebouwd
                df_plat['time_offset_in_seconds'] = matrix[:, 0]
                df_plat['bpm'] = matrix[:, 1]
                
                # 5. Gooi de oude object-kolom weg
                df_part = df_plat.drop(columns=['heart_rate_samples'])
                #print(df_part)
                dfs.append(df_part)
            else:
                print(f"XX - {f}")     
        else:
            print("YY")

    except Exception as e:
        print(f"Fout bij bestand {f}: {e}")
df = pd.concat(dfs, ignore_index=True)
df

START
0/396
1/396
2/396
3/396
4/396
XX - /Volumes/Extreme SSD/pq/parquet/heart_rate_sample/profile_id=10093440/part-0.parquet
5/396
6/396
7/396
8/396
9/396
10/396
11/396
XX - /Volumes/Extreme SSD/pq/parquet/heart_rate_sample/profile_id=10094145/part-0.parquet
12/396
13/396
14/396
XX - /Volumes/Extreme SSD/pq/parquet/heart_rate_sample/profile_id=10223670/part-0.parquet
15/396
16/396
17/396
18/396
XX - /Volumes/Extreme SSD/pq/parquet/heart_rate_sample/profile_id=10356376/part-0.parquet
19/396
XX - /Volumes/Extreme SSD/pq/parquet/heart_rate_sample/profile_id=10486108/part-0.parquet
20/396
21/396
22/396
23/396
XX - /Volumes/Extreme SSD/pq/parquet/heart_rate_sample/profile_id=10618017/part-0.parquet
24/396
25/396
26/396
27/396
XX - /Volumes/Extreme SSD/pq/parquet/heart_rate_sample/profile_id=10747962/part-0.parquet
28/396
29/396
30/396
31/396
32/396
33/396
34/396
XX - /Volumes/Extreme SSD/pq/parquet/heart_rate_sample/profile_id=10880358/part-0.parquet
35/396
XX - /Volumes/Extreme SSD/pq/par

,time,source,timezone_offset_in_seconds,profile_id,time_offset_in_seconds,bpm
0,2023-05-22 22:00:00,weconnect.garmin,NaN,10092816,15,71
1,2023-05-22 22:00:00,weconnect.garmin,NaN,10092816,30,71
2,2023-05-22 22:00:00,weconnect.garmin,NaN,10092816,45,71
3,2023-05-22 22:00:00,weconnect.garmin,NaN,10092816,60,71
4,2023-05-22 22:00:00,weconnect.garmin,NaN,10092816,75,71
...,...,...,...,...,...,...
561129730,2026-05-17 10:30:00,weconnect.garmin,0.0,9963594,720,116
561129731,2026-05-17 10:30:00,weconnect.garmin,0.0,9963594,735,116
561129732,2026-05-17 10:30:00,weconnect.garmin,0.0,9963594,750,116
561129733,2026-05-17 10:30:00,weconnect.garmin,0.0,9963594,765,116


In [39]:
ts = df.iloc[0]
type(ts)
ts.time, ts.time_offset_in_seconds, ts.bpm

(Timestamp('2023-05-22 22:00:00'), np.int64(15), np.int64(71))

In [ ]:
# 1. Voorbeeld-dataopbouw (simulatie van jouw lijst met tuples)
# Stel dat 'raw_data' de lijst is die vol staat met de tuples zoals in jouw voorbeeld
raw_data = [
    (pd.Timestamp('2024-11-20 12:18:57'), np.array([[15, 60], [30, 60], [45, 60]], dtype=object)),
    # ... meer opeenvolgende timestamps ...
]

# 2. Extraheer de features en zorg voor een vaste vorm (shape)
# We halen de 2D-arrays los en stacken ze in een 3D-numpy matrix
X_list = []
timestamps = []

for timestamp, array_2d in raw_data:
    # Converteer de 'object' array naar zuivere floats/integers
    matrix_clean = np.array(array_2d.tolist(), dtype=np.float32)
    X_list.append(matrix_clean)
    timestamps.append(timestamp)

# Maak er één grote 3D matrix van
# Vorm wordt: (aantal_timestamps, aantal_metingen_per_timestamp, aantal_kolommen)
# In jouw voorbeeld: (samples, 59, 2)
X_3d = np.array(X_list)

# 3. Normalisatie (Schaal de data tussen 0 en 1 voor de LSTM)
# Omdat MinMaxScaler alleen 2D data accepteert, flatten we het tijdelijk
samples, time_steps, features = X_3d.shape
X_2d_flat = X_3d.reshape(-1, features)

scaler = MinMaxScaler()
X_2d_scaled = scaler.fit_transform(X_2d_flat)

# Schaal het direct weer terug naar de originele 3D LSTM-vorm
X_lstm = X_2d_scaled.reshape(samples, time_steps, features)

print("Definitieve vorm voor Keras LSTM (X):", X_lstm.shape)
# Output: (samples, 59, 2)
